# 01 - Data preparation and quality

This notebook documents the input side of the project: how the raw Lending Club
extract is turned into a parquet table, how the stratified modeling sample is
built, and what the data-quality and leakage-control reports show.

**Target definition.** A loan is a *default* (`default_flag = 1`) when
`loan_status` is `Charged Off` or `Default`. Only loans with a **resolved**
outcome (`Fully Paid`, `Charged Off`, `Default`) are modeled; `Current` and
other in-flight statuses are excluded to avoid label immaturity.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.width', 160)
ROOT

In [ ]:
from src.config import Config
from src import data_prep, quality

cfg = Config.load()
if not cfg.path('modeling_parquet').exists():
    data_prep.build_accepted_parquet(force=False)
    data_prep.build_modeling_sample(force=False)
print('modeling sample:', cfg.path('modeling_parquet'))

In [ ]:
from src.pipeline import load_modeling_frame, train_oot_split

df = load_modeling_frame(cfg)
print('rows:', len(df), '| columns:', df.shape[1])
df[['loan_amnt', 'annual_inc', 'dti', 'grade', 'issue_year', 'default_flag']].head()

In [ ]:
rep = quality.build_reports(df)
display(quality.quality_summary(df))
df.groupby('issue_year')['default_flag'].agg(n_loans='size', default_rate='mean').round(4)

In [ ]:
display(rep['missingness'].head(15))
display(rep['outliers'].head(15))

## Leakage control

Every post-origination column (payments, recoveries, outstanding principal,
status dates, ...) is dropped before modeling. The table below reports how
strongly each of those columns is correlated with the realised default flag -
a high correlation would be a red flag that the variable carries future
information rather than origination information.

In [ ]:
display(quality.leakage_report(df).head(20))

### Notes

* `term` is kept as a categorical level (`36` / `60`), not as a number.
* Winsorisation bounds are learned on the training window only and reused
  out of time, so no future information leaks into the feature scaling.
* `grade`, `sub_grade` and `int_rate` are the lender's own risk assessment and
  are excluded from the main scorecard; they are only used for a benchmark
  sensitivity comparison in notebook 03.